In [1]:
# pip install openai chromadb python-dotenv requests pandas edgartools

In [2]:
import os
import time
import requests
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from edgar import Company, set_identity

load_dotenv()
client_ai = OpenAI()
set_identity("Isaiah Lacet isaiahlacet@gmail.com")

EMBEDDING_MODEL = "text-embedding-3-small"

chroma_client = chromadb.PersistentClient(path="vector_store")
collection = chroma_client.get_or_create_collection(name="filings")

In [3]:
HEADERS = {"User-Agent": "Isaiah Lacet isaiahlacet@gmail.com"}
_ticker_to_cik = None

def _load_ticker_map():
    global _ticker_to_cik
    if _ticker_to_cik is None:
        data = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS).json()
        _ticker_to_cik = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in data.values()}
    return _ticker_to_cik

def is_valid_ticker(ticker):
    return ticker.upper() in _load_ticker_map()

In [4]:
def get_latest_10k(ticker):
    ticker = ticker.upper()
    filings = [f for f in Company(ticker).get_filings(form="10-K") if f.form == "10-K"]
    if not filings:
        raise ValueError(f"No original (non-amended) 10-K filings found for '{ticker}'.")
    return filings[0]

In [5]:
def extract_sections(tenk, ticker):
    chunks = []
    for item in tenk.items:
        try:
            text = tenk[item]
        except Exception:
            continue
        if text and len(text.strip()) > 0:
            chunks.append({"ticker": ticker, "section": item, "text": text.strip()})
    return chunks

In [6]:
def split_long_text(text, max_chars=6000):
    if len(text) <= max_chars:
        return [text]
    parts = []
    paragraphs = text.split("\n")
    current = ""
    for para in paragraphs:
        if len(current) + len(para) + 1 > max_chars:
            if current:
                parts.append(current.strip())
            current = para
        else:
            current += "\n" + para if current else para
    if current:
        parts.append(current.strip())
    return parts

In [7]:
def looks_like_a_10k(chunks):
    for c in chunks:
        if "1A" in c["section"] and len(c["text"]) > 300:
            return True
    return False

In [8]:
def get_embedding(text, model=EMBEDDING_MODEL):
    resp = client_ai.embeddings.create(input=[text], model=model)
    return resp.data[0].embedding

def get_embeddings(texts, model=EMBEDDING_MODEL, batch_size=100):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client_ai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([e.embedding for e in resp.data])
    return all_embeddings

In [9]:
def is_ticker_indexed(ticker, accession_no=None):
    ticker = ticker.upper()
    existing = collection.get(where={"ticker": ticker}, limit=1)
    if len(existing["ids"]) == 0:
        return False
    if accession_no is None:
        return True
    indexed_accession = existing["metadatas"][0].get("accession_no")
    return indexed_accession == accession_no

def ensure_ticker_indexed(ticker):
    ticker = ticker.upper()

    if not is_valid_ticker(ticker):
        raise ValueError(f"'{ticker}' is not a recognized ticker in the SEC's ticker list.")

    filing = get_latest_10k(ticker)
    accession_no = filing.accession_no

    if is_ticker_indexed(ticker, accession_no=accession_no):
        return  # already indexed and it's the current filing

    existing = collection.get(where={"ticker": ticker})
    if existing["ids"]:
        print(f"Newer 10-K found for {ticker} — re-indexing...")
        collection.delete(ids=existing["ids"])

    print(f"Indexing latest 10-K for {ticker}...")
    tenk = filing.obj()

    raw_chunks = extract_sections(tenk, ticker)
    if not raw_chunks:
        raise ValueError(f"Parsed 0 sections from {ticker}'s 10-K.")
    if not looks_like_a_10k(raw_chunks):
        raise ValueError(f"Parsed sections for {ticker} don't look like a standard 10-K — manual review needed.")

    split_chunks = []
    for c in raw_chunks:
        pieces = split_long_text(c["text"])
        for i, piece in enumerate(pieces):
            split_chunks.append({
                "ticker": c["ticker"],
                "section": c["section"] if len(pieces) == 1 else f"{c['section']} (part {i+1})",
                "text": piece
            })

    texts = [c["text"] for c in split_chunks]
    embeddings = get_embeddings(texts)
    ids = [f"{ticker}_{i}" for i in range(len(split_chunks))]
    metadatas = [{"ticker": c["ticker"], "section": c["section"], "accession_no": accession_no} for c in split_chunks]

    collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
    print(f"Indexed {len(split_chunks)} chunks for {ticker} (accession {accession_no}).")

In [10]:
def extract_tickers(question, model="gpt-4o-mini"):
    prompt = f"""Extract the stock ticker symbol(s) for any publicly traded company mentioned in this question.
Return ONLY the ticker symbols, comma-separated, uppercase, nothing else.
If no company is mentioned, return NONE.

Question: {question}"""
    resp = client_ai.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    raw = resp.choices[0].message.content.strip()
    if raw == "NONE":
        return []
    return [t.strip().upper() for t in raw.split(",")]

def validate_tickers(tickers):
    return [t for t in tickers if is_valid_ticker(t)]

In [11]:
def retrieve(query, query_embedding=None, n_per_ticker=20, tickers=None):
    if not tickers:
        raise ValueError("retrieve() requires at least one ticker in `tickers`.")
    if query_embedding is None:
        query_embedding = get_embedding(query)

    all_results = []
    for ticker in tickers:
        ensure_ticker_indexed(ticker)
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_per_ticker,
            where={"ticker": ticker.upper()}
        )
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            all_results.append({"text": doc, "ticker": meta["ticker"], "section": meta["section"]})
    return all_results

def ask(question, query_embedding=None, relevant_tickers=None, n_per_ticker=20, model="gpt-4o-mini"):
    if not relevant_tickers:
        relevant_tickers = validate_tickers(extract_tickers(question))
        if not relevant_tickers:
            raise ValueError("Couldn't identify a valid company/ticker in the question. Try passing relevant_tickers explicitly.")
        print(f"Auto-detected ticker(s): {relevant_tickers}")

    tickers = tuple(t.upper() for t in relevant_tickers)
    chunks = retrieve(question, query_embedding=query_embedding, n_per_ticker=n_per_ticker, tickers=tickers)
    context = "\n\n".join(f"[{c['ticker']} - {c['section']}]\n{c['text']}" for c in chunks)

    prompt = f"""Answer the question using only the context below, from SEC 10-K filings.

Rules:
- Paraphrase in your own words. No quotation marks, no verbatim quotes.
- Favor specific facts (numbers, named events, past-tense admissions, hedges like "no assurance"/"may not") over generic risk language, unless the question specifically asks about risks — in that case, include risk statements from the context even if they are narrative rather than numeric, as long as they directly address the risk being asked about.
- Don't merge facts from different context chunks into one statement or imply a causal link between them unless the text states it directly.
- Only state what the context directly supports.
- If the answer isn't in the context, say so.
- If the context presents information as a list, category breakdown, or enumeration (e.g., competitor types, business segments, risk categories, product lines), include every item present in the context, at every level of nesting. If a category itself breaks down into sub-items or sub-lines of business (e.g., a segment that contains multiple named lines of business), enumerate those sub-items too. Do not condense a list or sub-list down to a representative subset — completeness of enumerations is a hard requirement, not a style choice, and applies recursively to nested breakdowns.
- Format the entire answer as a bulleted list. Each bullet must contain exactly one fact or claim. Do not include citations or section references in the answer text — sources are reported separately.

Context:
{context}

Question: {question}

Answer:"""

    for attempt in range(5):
        try:
            resp = client_ai.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            return resp.choices[0].message.content, chunks
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt * 5
                print(f"Rate limited, waiting {wait}s (attempt {attempt+1}/5)...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Failed after 5 retries due to rate limiting")

Legal proceedings (Auto-Detect)

In [12]:
question = "What legal proceedings or litigation does Tesla disclose in its 10-K?"
answer, sources = ask(question)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

Auto-detected ticker(s): ['TSLA']
ANSWER:
 - Between October 17, 2018, and March 8, 2021, seven derivative lawsuits were filed in the Delaware Court of Chancery against Mr. Musk and Tesla's board of directors related to a potential going private transaction and certain Twitter posts by Mr. Musk.
- Several of the derivative lawsuits were consolidated, and all non-consolidated cases were dismissed with prejudice.
- A stipulation for dismissal with prejudice of the consolidated case was filed on December 24, 2025, pending court approval.
- Two additional derivative lawsuits were filed on October 25, 2018, and February 11, 2019, in the U.S. District Court for the District of Delaware, which were also consolidated and dismissed with prejudice on April 25, 2025.
- On October 21, 2022, a lawsuit was filed in the Delaware Court of Chancery by a purported shareholder alleging breaches of fiduciary duties by board members regarding the 2018 SEC settlement, seeking corporate governance reforms, u

Competition (Auto-Detect)

In [13]:
question = "Who does Amazon identify as its main competitors?"
answer, sources = ask(question)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

Auto-detected ticker(s): ['AMZN']
ANSWER:
 - Amazon faces competition from a variety of sources, including:
  - Physical retailers
  - E-commerce retailers
  - Omnichannel retailers
  - Publishers
  - Vendors
  - Distributors
  - Manufacturers
  - Producers of the products it offers
  - Publishers, producers, and distributors of physical, digital, and interactive media
  - Web search engines
  - Comparison shopping websites
  - Social networks
  - Web portals
  - Virtual assistants
  - Companies providing e-commerce services (website development, hosting, omnichannel sales, inventory and supply chain management, advertising, fulfillment, customer service, payment processing)
  - Companies providing fulfillment and logistics services
  - Companies offering information technology services or products (including cloud-based infrastructure and artificial intelligence tools)
  - Companies designing, manufacturing, marketing, or selling consumer electronics and communication devices
  - Comp

Disclosure (Ticker specified)

In [14]:
question = "What does Costco disclose about risks related to membership fee renewal rates?"
answer, sources = ask(question, relevant_tickers=["COST"])

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

ANSWER:
 - Membership loyalty and growth are crucial for Costco's business.
- The company’s profitability is significantly influenced by the growth of its membership base, the penetration of Executive memberships, and the maintenance of high renewal rates.
- Damage to Costco's brands or reputation could negatively impact comparable sales and member trust.
- A decline in renewal rates could adversely affect net sales and membership fee revenue.
- The renewal rate for memberships in the U.S. and Canada was 92.3% at the end of 2025, while the worldwide renewal rate was 89.8%.

SOURCES USED:
- COST: Item 1 (part 3)
- COST: Item 8 (part 8)
- COST: Item 16
- COST: Item 8 (part 4)
- COST: Item 8 (part 14)
- COST: Item 5
- COST: Item 4
- COST: Item 11
- COST: Item 1 (part 2)
- COST: Item 1 (part 1)
- COST: Item 13
- COST: Item 8 (part 13)
- COST: Item 12
- COST: Item 10
- COST: Item 8 (part 2)
- COST: Item 1A (part 3)
- COST: Item 8 (part 1)
- COST: Item 1A (part 8)
- COST: Item 1A (part 1)
- 

Business segments (Ticker specified)

In [15]:
question = "How does Disney break down its business into reportable segments?"
answer, sources = ask(question, relevant_tickers=["DIS"])

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

ANSWER:
 - The Walt Disney Company categorizes its business into three main segments: Entertainment, Sports, and Experiences.
  
**Entertainment Segment:**
- This segment includes non-sports focused global film and episodic content production and distribution activities.
- Lines of business within Entertainment:
  - **Linear Networks:**
    - Domestic: ABC Television Network, Disney, Freeform, FX, and National Geographic channels, and eight owned ABC television stations.
    - International: Disney, FX, and National Geographic channels.
    - A 50% equity investment in A+E Global Media.
  - **Direct-to-Consumer:**
    - Disney+: A global streaming service offering general entertainment and family programming.
    - Hulu: A U.S. streaming service providing general entertainment and live TV.
  - **Content Sales/Licensing:**
    - Theatrical distribution, sale/licensing of content to TV/VOD services, home entertainment distribution, intersegment revenue allocation, staging and licensing o

Executive compensation (n_per_ticker tuned)

In [16]:
question = "What does NVIDIA disclose about its employee compensation structure?"
answer, sources = ask(question, n_per_ticker=50)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

Auto-detected ticker(s): ['NVDA']
ANSWER:
 - NVIDIA's employee compensation structure includes stock-based compensation, which is a significant component of total compensation.
- As of January 25, 2026, the aggregate unearned stock-based compensation expense was $14.8 billion, expected to be recognized over a weighted average period of 2.3 years for RSUs, PSUs, and market-based PSUs, and 0.9 years for the Employee Stock Purchase Plan (ESPP).
- In fiscal year 2026, stock-based compensation expense totaled $6.4 billion, with $4.7 billion allocated to research and development, and $1.4 billion to sales, general, and administrative expenses.
- The company grants various types of equity awards under its equity incentive plans, including RSUs, PSUs, market-based PSUs, and stock purchase rights.
- The Amended and Restated 2007 Equity Incentive Plan allows for the issuance of various stock options and awards, with 192 million shares authorized for issuance and 1.3 billion shares available for 

Risk Factors (n_per_ticker tuned)

In [17]:
question = "What does Nike say about the risk of counterfeit products harming its brand?"
answer, sources = ask(question, n_per_ticker=30)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

Auto-detected ticker(s): ['NKE']
ANSWER:
 - Nike periodically discovers counterfeit reproductions of its products or products that infringe on its intellectual property rights.
- If Nike is unsuccessful in enforcing its intellectual property rights, continued sales of counterfeit products could adversely affect its sales and brand.
- A shift in consumer preference away from Nike's products could result from the presence of counterfeit goods in the market.
- Nike may face significant expenses and liability in connection with the protection of its intellectual property rights, including defending against claims of infringement.

SOURCES USED:
- NKE: Item 1A (part 2)
- NKE: Item 1 (part 4)
- NKE: Item 1A (part 1)
- NKE: Item 1A (part 12)
- NKE: Item 1C
- NKE: Item 1A (part 3)
- NKE: Item 1 (part 3)
- NKE: Item 1A (part 17)
- NKE: Item 1A (part 10)
- NKE: Item 1 (part 1)
- NKE: Item 1A (part 6)
- NKE: Item 1A (part 9)
- NKE: Item 7 (part 1)
- NKE: Item 1A (part 8)
- NKE: Item 1A (part 7)
-